# Multivariate Ehrhart polynomials

Examples of computing $E(n_1,\ldots,n_k)=\#((n_1P_1+\cdots+n_kP_k)\cap\mathbb Z^d)$ for lattice polytopes $P_1,\ldots,P_k$.


In [ ]:
from sage.all import QQ
from mixed_ehrhart import (
    homogeneous_part,
    lattice_count,
    minkowski_sum_scaled,
    multivariate_ehrhart,
    multivariate_ehrhart_from_vertices,
    poly_from_vertices,
)

## Perpendicular segments

The Minkowski sum of $a*[0,e_1]$ and $b*[0,e_2]$ is an $a$-by-$b$ rectangle with $(a+1)(b+1)$ lattice points. The returned pair is a polynomial and its coefficients in the binomial basis.

In [ ]:
P = poly_from_vertices([(0, 0), (1, 0)])
Q = poly_from_vertices([(0, 0), (0, 1)])
ehrhart, differences = multivariate_ehrhart([P, Q], names=["a", "b"])
a, b = ehrhart.parent().gens()
assert ehrhart == (a + 1) * (b + 1)
assert differences[(1, 1)] == 1
assert ehrhart(0, 0) == 1
ehrhart

Check the polynomial against direct lattice-point counts at a few values:

In [ ]:
for i in range(4):
    for j in range(4):
        rectangle = minkowski_sum_scaled([P, Q], [i, j])
        assert ehrhart(i, j) == lattice_count(rectangle)

translated = poly_from_vertices([(5, 7), (6, 7)])
zero_sum = minkowski_sum_scaled([translated], [0])
assert lattice_count(zero_sum) == 1
assert zero_sum.contains((0, 0))

## Repeated standard triangle: distinguish the two bases

For $T=\operatorname{conv}((0,0),(1,0),(0,1))$, one has
$aT+bT=(a+b)T$. Therefore

$$E(a,b)=\frac{(a+b)^2+3(a+b)+2}{2}.$$

The forward-difference coefficients are those of
$\binom{a}{r}\binom{b}{s}$, not $a^r b^s$. The code expands this basis when constructing the returned ordinary polynomial.

In [ ]:
triangle_vertices = [(0, 0), (1, 0), (0, 1)]
triangle = poly_from_vertices(triangle_vertices)
triangle_ehrhart, triangle_differences = multivariate_ehrhart_from_vertices(
    [triangle_vertices, triangle_vertices], names=["a", "b"]
)
a, b = triangle_ehrhart.parent().gens()
expected = ((a + b)**2 + 3 * (a + b) + 2) / 2
assert triangle_ehrhart == expected
assert triangle_differences[(2, 0)] == 1
assert triangle_differences[(1, 0)] == 2
assert triangle_ehrhart.monomial_coefficient(a**2) == QQ(1) / 2
assert triangle_ehrhart.monomial_coefficient(a) == QQ(3) / 2
triangle_ehrhart, triangle_differences

`homogeneous_part` extracts ordinary monomials of a given total degree.

In [ ]:
quadratic = homogeneous_part(triangle_ehrhart, 2)
linear = homogeneous_part(triangle_ehrhart, 1)
assert quadratic == (a + b)**2 / 2
assert linear == 3 * (a + b) / 2
for i in range(4):
    for j in range(4):
        assert triangle_ehrhart(i, j) == lattice_count(
            minkowski_sum_scaled([triangle, triangle], [i, j])
        )
quadratic, linear

## Optional: supply an exact counter

Use `count=...` to select a different exact counting method. The callable takes a SageMath polytope and returns a nonnegative integer. Large polytopes should use an appropriate counting backend.

In [ ]:
def enumerating_counter(poly):
    return len(poly.integral_points())

checked, _ = multivariate_ehrhart(
    [P, Q], names=["a", "b"], count=enumerating_counter
)
assert checked == ehrhart

The number of interpolation sample points is $\binom{D+k}{k}$, where
$D=\dim(P_1+\cdots+P_k)$. Each sample requires a lattice count, so examples with many inputs or large coordinates can still be expensive.